In [18]:
import nbformat
# from nbformat.v4 import new_notebook, new_code_cell
import pandas as pd
from nbformat.v4 import new_notebook, new_code_cell, new_markdown_cell

In [19]:
df = pd.read_excel("verified_questions.xlsx")
# df = df[df.index>50]
df = df[df["OpenTarget_compability"]=="yes"]
df.reset_index(inplace=True, drop=True)
questions = list(df["Question"])

In [20]:
# # ----------------------------
# # QUESTIONS
# # ----------------------------
# questions = [
#     "Which diseases are associated with BRAF?",
#     "List genes linked to amyotrophic lateral sclerosis.",
# ]


In [21]:


# ----------------------------
# NOTEBOOK
# ----------------------------
nb = new_notebook(cells=[])

# ----------------------------
# CELL 1 — Imports & config
# ----------------------------
nb.cells.append(
    new_code_cell(
        """
import asyncio
import time
import json
import websockets
import requests
import pandas as pd
from pathlib import Path
from opentarget_utility import retrieve_opentarget


"""
    )
)


nb.cells.append(
    new_code_cell(
        """


model = "OpenTarget"
dct_result = {}
dct_result[model] = {}

MAX_RETRIES = 3
RUNS = range(1, 6)
"""
    )
)

# ----------------------------
# QUESTION CELLS (Markdown + Code)
# ----------------------------
for i, q in enumerate(questions, start=1):

    # ---- Markdown cell (Question header) ----
    nb.cells.append(
        new_markdown_cell(
            f"""
## Question {i}

**{q}**
"""
        )
    )

    # ---- Code cell (execution logic) ----
    nb.cells.append(
        new_code_cell(
            f"""
question = {q!r}


dct_result[model].setdefault(question, {{}})



for run_number in RUNS:
    print(f"\\n Starting run {{run_number}} for query: {{question}}")

    for attempt in range(1, MAX_RETRIES + 1):
        print(f"  Attempt {{attempt}}/{{MAX_RETRIES}}")

        

        try:
            start_time = time.perf_counter()
            df = await retrieve_opentarget(question)
            latency = time.perf_counter() - start_time

            if not isinstance(df, pd.DataFrame):
                raise ValueError("Returned object is not a DataFrame")
            if df.empty:
                raise ValueError("Returned DataFrame is empty")

            dct_result[model][question][run_number] = {{
                "rows": df.shape[0],
                "latency": latency,
                "dataframe": df,
            }}

            print(
                f" Run {{run_number}} completed | "
                f"rows={{df.shape[0]}} | latency={{latency:.3f}}s"
            )
            break

        except Exception as e:
            latency = time.perf_counter() - start_time
            print(f" Attempt {{attempt}} failed after {{latency:.3f}}s: {{e}}")

            if attempt == MAX_RETRIES:
                print(f" Run {{run_number}} failed after {{MAX_RETRIES}} attempts")

    print("-" * 60)
"""
        )
    )


In [22]:
# ----------------------------
# WRITE NOTEBOOK
# ----------------------------
with open("biochirp_opentarget_questions_response.ipynb", "w", encoding="utf-8") as f:
    nbformat.write(nb, f)

print("✅ Notebook generated: biochirp_opentarget_questions_response.ipynb")


✅ Notebook generated: biochirp_opentarget_questions_response.ipynb
